# 🤖 Function Calling with LLMs — A Complete Beginner's Guide

---

## What is Function Calling?

Function calling (also called **Tool Use**) is a powerful feature that lets an LLM decide **when to call external functions** to get real-world data — instead of guessing from stale training data.

### 🌍 The 5-Step Flow (from the diagram)

```
User: "What's the weather in Paris?"
  ↓
Step 1: Developer sends tool definitions + user message → Model
  ↓
Step 2: Model responds with a tool call → get_weather("paris")
  ↓
Step 3: Developer executes the real function (calls Tavily API)
  ↓
Step 4: Developer sends results back → Model
  ↓
Step 5: Model responds → "It's currently 14°C in Paris."
```

### 🔑 Key Insight
> The LLM **never calls the function itself**. It only decides *which* function and *what arguments*. **You** (the developer) run the function and feed results back.


Every tool in this notebook calls the **Tavily Search API** — a real-time AI search engine. You'll get live, accurate answers.

---

### 📋 What You'll Learn
1. How to define tools/functions for an LLM
2. How the LLM routes queries to the right tool
3. How to execute real API calls and return results
4. The full 5-step flow, step by step
5. How to build and register your own tool

**Model:** `llama3.2:1b` via Ollama &nbsp;|&nbsp; **Search:** Tavily API (real-time web search)

---
## 📦 Cell 1: Install Dependencies

We need:
- `ollama` — to talk to the local LLM
- `tavily-python` — to make real API calls to Tavily Search
- `requests` — for any raw HTTP calls

**Before running:** Make sure Ollama is running and the model is pulled:
```bash
ollama serve
ollama pull llama3.2:1b
```

In [1]:
!pip install ollama tavily-python requests --quiet

print("✅ All libraries installed!")
print("📌 Ollama must be running:  ollama serve")
print("📌 Model must be pulled:    ollama pull llama3.2:1b")
print("📌 Get a free Tavily key:   https://tavily.com  (free tier = 1000 searches/month)")

✅ All libraries installed!
📌 Ollama must be running:  ollama serve
📌 Model must be pulled:    ollama pull llama3.2:1b
📌 Get a free Tavily key:   https://tavily.com  (free tier = 1000 searches/month)


---
## 🔑 Cell 2: Configure API Keys & Verify Connections

Set your **Tavily API key** here. Get one free at [https://tavily.com](https://tavily.com).

Then we verify both Ollama and Tavily are reachable before proceeding.

In [2]:
import ollama
import json
from tavily import TavilyClient

# ─────────────────────────────────────────────
# 🔑  SET YOUR TAVILY API KEY HERE
TAVILY_API_KEY = "YOUR-API-KEY"   # <── Replace with your real key
# ─────────────────────────────────────────────

MODEL = "llama3.2:1b"

print("=" * 60)
print("🔍 Verifying connections...")
print("=" * 60)

# ── Check Ollama ──
try:
    models = ollama.list()
    model_names = [m.model for m in models.models]
    print(f"\n✅ Ollama connected!")
    print(f"   Available models: {model_names}")
    if any(MODEL in n for n in model_names):
        print(f"   🎯 '{MODEL}' is ready!")
    else:
        print(f"   ⚠️  '{MODEL}' not found — run: ollama pull {MODEL}")
except Exception as e:
    print(f"\n❌ Ollama not reachable: {e}")

# ── Check Tavily ──
print()
try:
    tavily = TavilyClient(api_key="REDACTED_TAVILY_KEY")
    test = tavily.search("current date", max_results=1)
    print(f"✅ Tavily connected!")
    print(f"   Test search returned {len(test.get('results', []))} result(s)")
except Exception as e:
    print(f"❌ Tavily error: {e}")
    print("   Double-check your API key at https://app.tavily.com")

print("\n" + "=" * 60)

🔍 Verifying connections...

✅ Ollama connected!
   Available models: ['llama3.2:1b', 'gemma:2b']
   🎯 'llama3.2:1b' is ready!

✅ Tavily connected!
   Test search returned 1 result(s)



---
## 🛠️ Cell 3: Define Real Tool Functions (Powered by Tavily)



Every function below calls the **Tavily Search API** to fetch live, real-time information from the web. Tavily is an AI-optimized search engine that returns clean, structured results — perfect for feeding into an LLM.

| Function | What it searches | Example |
|---|---|---|
| `get_weather` | Current weather for a city | "weather in Paris right now" |
| `get_stock_price` | Live stock data | "AAPL Apple stock price today" |
| `search_news` | Latest news on any topic | "latest AI news" |
| `calculate` | Safe math evaluation | `"15 * 8 + 42"` → Python eval |

> 💡 `calculate` uses Python's `eval()` — no network call needed for pure math!

In [3]:
from tavily import TavilyClient

tavily = TavilyClient(api_key="REDACTED_TAVILY_KEY")


# ──────────────────────────────────────────────────────────
# Tool 1: get_weather — real-time weather via Tavily search
# ──────────────────────────────────────────────────────────
def get_weather(location: str, unit: str = "celsius") -> dict:
    """
    Fetches real-time weather for a city using Tavily web search.
    Returns structured data extracted from live weather sources.
    """
    query = f"current weather in {location} today temperature {unit}"
    print(f"   🌐 Tavily search: \"{query}\"")

    response = tavily.search(
        query=query,
        search_depth="basic",
        max_results=3,
        include_answer=True   # Tavily returns a synthesized answer
    )

    answer   = response.get("answer", "Weather data unavailable")
    sources  = [r["url"] for r in response.get("results", [])[:2]]

    return {
        "location": location.title(),
        "unit": unit,
        "weather_summary": answer,
        "sources": sources
    }


# ──────────────────────────────────────────────────────────
# Tool 2: get_stock_price — live stock data via Tavily
# ──────────────────────────────────────────────────────────
def get_stock_price(symbol: str) -> dict:
    """
    Fetches the current stock price and market data using Tavily.
    """
    query = f"{symbol} stock price today current market value"
    print(f"   🌐 Tavily search: \"{query}\"")

    response = tavily.search(
        query=query,
        search_depth="basic",
        max_results=3,
        include_answer=True
    )

    answer  = response.get("answer", "Stock data unavailable")
    sources = [r["url"] for r in response.get("results", [])[:2]]

    return {
        "symbol": symbol.upper(),
        "market_data_summary": answer,
        "sources": sources
    }


# ──────────────────────────────────────────────────────────
# Tool 3: search_news — latest news on any topic via Tavily
# ──────────────────────────────────────────────────────────
def search_news(topic: str, max_results: int = 3) -> dict:
    """
    Fetches the latest news headlines on any topic using Tavily.
    """
    query = f"latest news about {topic} 2025"
    print(f"   🌐 Tavily search: \"{query}\"")

    response = tavily.search(
        query=query,
        search_depth="basic",
        max_results=max_results,
        include_answer=True,
        topic="news"
    )

    answer   = response.get("answer", "")
    articles = [
        {"title": r.get("title", ""), "url": r.get("url", ""),
         "snippet": r.get("content", "")[:200]}
        for r in response.get("results", [])
    ]

    return {
        "topic": topic,
        "summary": answer,
        "articles": articles
    }


# ──────────────────────────────────────────────────────────
# Tool 4: calculate — safe Python math eval (no API needed)
# ──────────────────────────────────────────────────────────
def calculate(expression: str) -> dict:
    """
    Evaluates a mathematical expression safely using Python.
    No network call — pure computation.
    """
    try:
        safe_env = {"__builtins__": None,
                    "abs": abs, "round": round, "pow": pow,
                    "max": max, "min": min, "sum": sum}
        result = eval(expression, safe_env)
        return {"expression": expression, "result": result, "success": True}
    except Exception as e:
        return {"expression": expression, "error": str(e), "success": False}


# Function registry
FUNCTION_REGISTRY = {
    "get_weather":    get_weather,
    "get_stock_price": get_stock_price,
    "search_news":    search_news,
    "calculate":      calculate,
}

print("=" * 60)
print("🛠️  Real Tool Functions Defined — Testing Each One...")
print("=" * 60)

print("\n🌦️  get_weather('Paris'):")
r = get_weather("Paris")
print(f"   Summary: {r['weather_summary'][:120]}...")
print(f"   Sources: {r['sources']}")

print("\n📈 get_stock_price('AAPL'):")
r = get_stock_price("AAPL")
print(f"   Summary: {r['market_data_summary'][:120]}...")

print("\n📰 search_news('artificial intelligence'):")
r = search_news("artificial intelligence", max_results=2)
print(f"   Summary: {r['summary'][:120]}...")
print(f"   Articles: {len(r['articles'])} found")

print("\n🔢 calculate('15 * 8 + 42'):")
r = calculate("15 * 8 + 42")
print(f"   Result: {r}")

print("\n✅ All tools working with REAL data!")

🛠️  Real Tool Functions Defined — Testing Each One...

🌦️  get_weather('Paris'):
   🌐 Tavily search: "current weather in Paris today temperature celsius"
   Summary: Today in Paris, the temperature is 27°C, according to Météo-France. This is part of a severe high-temperature warning is...
   Sources: ['https://www.weatherapi.com/', 'https://polymarket.com/event/highest-temperature-in-paris-on-may-30-2026']

📈 get_stock_price('AAPL'):
   🌐 Tavily search: "AAPL stock price today current market value"
   Summary: As of today, the AAPL stock price is $312.06, down 0.45% or 0.14%. This reflects a minor decline from the opening price....

📰 search_news('artificial intelligence'):
   🌐 Tavily search: "latest news about artificial intelligence 2025"
   Summary: In 2025, the unveiling of Android XR prototypes and a new Gemini video AI by a major tech company at I/O sparked a signi...
   Articles: 2 found

🔢 calculate('15 * 8 + 42'):
   Result: {'expression': '15 * 8 + 42', 'result': 162, 'succe

---
## 📖 Cell 4: Define Tool Schemas (The LLM's Manual)

The LLM never sees your Python code. It only reads the **schema** — a structured JSON description of each tool.

A schema has 4 parts:
```
name        → what to call the function
description → WHEN to use it (the most critical part!)
parameters  → what inputs to pass
required    → which inputs are mandatory
```

> 💡 **Bad description** = wrong tool chosen. **Good description** = perfect routing. The description is everything!

In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": (
                "Get the current real-time weather for a city or location. "
                "Use this when the user asks about weather conditions, temperature, "
                "rain, humidity, climate, or whether to bring an umbrella."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City or location name, e.g. 'Paris', 'New York', 'Tokyo'"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit. Default is celsius."
                    }
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": (
                "Get the current live stock price and market data for a company. "
                "Use when the user asks about stock prices, share values, market cap, "
                "or financial performance of a publicly traded company."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "symbol": {
                        "type": "string",
                        "description": "Stock ticker symbol, e.g. 'AAPL' for Apple, 'TSLA' for Tesla, 'GOOGL' for Google"
                    }
                },
                "required": ["symbol"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_news",
            "description": (
                "Search for the latest real-time news and recent events on any topic. "
                "Use when the user asks about recent news, current events, "
                "latest developments, or anything that may have happened recently."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "The topic or subject to search news for, e.g. 'AI research', 'SpaceX', 'climate change'"
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Number of news articles to return. Default is 3."
                    }
                },
                "required": ["topic"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": (
                "Perform precise mathematical calculations. Use when the user asks "
                "for arithmetic, algebra, percentages, or any numerical computation "
                "where exact accuracy is required."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A Python math expression, e.g. '25 * 4 + 10' or 'pow(2, 10)'"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

print("=" * 60)
print("📖 Tool Schemas Defined")
print("=" * 60)
print(f"\n   Total tools registered: {len(tools)}")
for i, t in enumerate(tools, 1):
    fn = t["function"]
    params = list(fn["parameters"]["properties"].keys())
    required = fn["parameters"].get("required", [])
    data_source = "🌐 Tavily API" if fn['name'] != 'calculate' else "🐍 Python eval"
    print(f"\n   Tool {i}: {fn['name']}")
    print(f"   Data source:  {data_source}")
    print(f"   Parameters:   {params}")
    print(f"   Required:     {required}")

print("\n" + "=" * 60)
print("✅ Schemas ready — the LLM now knows when and how to use each tool!")

📖 Tool Schemas Defined

   Total tools registered: 4

   Tool 1: get_weather
   Data source:  🌐 Tavily API
   Parameters:   ['location', 'unit']
   Required:     ['location']

   Tool 2: get_stock_price
   Data source:  🌐 Tavily API
   Parameters:   ['symbol']
   Required:     ['symbol']

   Tool 3: search_news
   Data source:  🌐 Tavily API
   Parameters:   ['topic', 'max_results']
   Required:     ['topic']

   Tool 4: calculate
   Data source:  🐍 Python eval
   Parameters:   ['expression']
   Required:     ['expression']

✅ Schemas ready — the LLM now knows when and how to use each tool!


---
## ⚙️ Cell 5: The 5-Step Function Calling Engine

This is the **orchestration layer** — it implements the exact 5-step diagram:

```
Step 1 → Send tools + user message to model
Step 2 → Model decides: call a tool? Which one? What args?
Step 3 → Execute the real Python function (calls Tavily)
Step 4 → Send tool result + full history back to model
Step 5 → Model synthesizes a final human-readable answer
```

In [5]:
def print_step(step_num: int, title: str, content: str = ""):
    icons = {1: "📤", 2: "🤖", 3: "⚙️", 4: "📬", 5: "💬"}
    icon = icons.get(step_num, "→")
    print(f"\n{'─'*58}")
    print(f"  {icon}  STEP {step_num}: {title}")
    print(f"{'─'*58}")
    if content:
        print(content)


def run_function_calling(user_message: str, verbose: bool = True) -> str:
    """
    Full function calling pipeline with real Tavily-powered tools.
    Mirrors the 5-step diagram exactly.
    """
    messages = [{"role": "user", "content": user_message}]

    # ── STEP 1 ───────────────────────────────────────────────
    if verbose:
        print_step(1, "Tool Definitions + Message → Model",
            f"  User: \"{user_message}\"\n"
            f"  Tools sent: {[t['function']['name'] for t in tools]}")

    response = ollama.chat(model=MODEL, messages=messages, tools=tools)
    assistant_msg = response["message"]
    tool_calls    = assistant_msg.get("tool_calls", [])

    # ── STEP 2 ───────────────────────────────────────────────
    if verbose:
        if tool_calls:
            calls_info = ""
            for tc in tool_calls:
                fn_name = tc["function"]["name"]
                fn_args = tc["function"]["arguments"]
                calls_info += f"  🔧 {fn_name}({json.dumps(fn_args)})\n"
            print_step(2, "Model → Tool Call Decision", calls_info)
        else:
            print_step(2, "Model → Direct Answer (no tool needed)",
                "  ℹ️  Model answered from its own knowledge — no API call needed.")

    # No tool calls → return direct answer
    if not tool_calls:
        answer = assistant_msg.get("content", "No response.")
        if verbose:
            print_step(5, "Final Response", f"  {answer}")
        return answer

    messages.append({
        "role": "assistant",
        "content": assistant_msg.get("content", ""),
        "tool_calls": tool_calls
    })

    # ── STEP 3 + 4 ───────────────────────────────────────────
    for tc in tool_calls:
        fn_name = tc["function"]["name"]
        fn_args = tc["function"]["arguments"]

        if verbose:
            print_step(3, f"Execute Real Function → {fn_name}",
                f"  Calling: {fn_name}({json.dumps(fn_args)})\n"
                f"  Hitting Tavily API now...")

        fn_result = FUNCTION_REGISTRY.get(fn_name, lambda **k: {"error": "unknown"})(** fn_args)
        result_str = json.dumps(fn_result)

        if verbose:
            preview = result_str[:300] + ("..." if len(result_str) > 300 else "")
            print(f"  ✅ Live result: {preview}")

        messages.append({"role": "tool", "content": result_str})

    if verbose:
        print_step(4, "Tool Results → Model",
            f"  Sending {len(messages)} messages (full history + live results) back...")

    # ── STEP 5 ───────────────────────────────────────────────
    final = ollama.chat(model=MODEL, messages=messages)
    final_text = final["message"]["content"]

    if verbose:
        print_step(5, "Final Response ← Model", f"  {final_text}")

    return final_text


print("=" * 60)
print("⚙️  Function Calling Engine Ready!")
print("=" * 60)
print("""
  Step 1 → tools + message ──────────────────→ Model
  Step 2 ←── model picks tool + args ─────────
  Step 3 → execute Python fn (calls Tavily) ──
  Step 4 → live result ──────────────────────→ Model
  Step 5 ←── final human-readable answer ─────
""")

⚙️  Function Calling Engine Ready!

  Step 1 → tools + message ──────────────────→ Model
  Step 2 ←── model picks tool + args ─────────
  Step 3 → execute Python fn (calls Tavily) ──
  Step 4 → live result ──────────────────────→ Model
  Step 5 ←── final human-readable answer ─────



---
## 🌦️ Cell 6: Demo 1 — Live Weather (Exact Diagram Example)

Reproducing the exact example from the diagram: **"What's the weather in Paris?"**

Watch each step of the flow — and notice the result comes from a **real Tavily API call**, not a dictionary.

In [6]:
print("=" * 60)
print("🌦️  DEMO 1: Live Weather — Exact Diagram Example")
print("=" * 60)

query = "What's the weather in Paris?"
print(f"\n🗣️  User: \"{query}\"")

answer = run_function_calling(query, verbose=True)

print("\n" + "=" * 60)
print("🎯 FINAL ANSWER:")
print(f"   {answer}")
print("=" * 60)
print("✅ Powered by: Tavily real-time search → llama3.2:1b synthesis")

🌦️  DEMO 1: Live Weather — Exact Diagram Example

🗣️  User: "What's the weather in Paris?"

──────────────────────────────────────────────────────────
  📤  STEP 1: Tool Definitions + Message → Model
──────────────────────────────────────────────────────────
  User: "What's the weather in Paris?"
  Tools sent: ['get_weather', 'get_stock_price', 'search_news', 'calculate']

──────────────────────────────────────────────────────────
  🤖  STEP 2: Model → Tool Call Decision
──────────────────────────────────────────────────────────
  🔧 get_weather({"location": "Paris"})


──────────────────────────────────────────────────────────
  ⚙️  STEP 3: Execute Real Function → get_weather
──────────────────────────────────────────────────────────
  Calling: get_weather({"location": "Paris"})
  Hitting Tavily API now...
   🌐 Tavily search: "current weather in Paris today temperature celsius"
  ✅ Live result: {"location": "Paris", "unit": "celsius", "weather_summary": "Today in Paris, the temperature i

---
## 📈 Cell 7: Demo 2 — Live Stock Price

The model should automatically route this to `get_stock_price`. The result is fetched **live** from financial web sources via Tavily.

In [7]:
print("=" * 60)
print("📈 DEMO 2: Live Stock Price")
print("=" * 60)

query = "What is the current stock price of Apple (AAPL)?"
print(f"\n🗣️  User: \"{query}\"")

answer = run_function_calling(query, verbose=True)

print("\n" + "=" * 60)
print("🎯 FINAL ANSWER:")
print(f"   {answer}")
print("=" * 60)

📈 DEMO 2: Live Stock Price

🗣️  User: "What is the current stock price of Apple (AAPL)?"

──────────────────────────────────────────────────────────
  📤  STEP 1: Tool Definitions + Message → Model
──────────────────────────────────────────────────────────
  User: "What is the current stock price of Apple (AAPL)?"
  Tools sent: ['get_weather', 'get_stock_price', 'search_news', 'calculate']

──────────────────────────────────────────────────────────
  🤖  STEP 2: Model → Tool Call Decision
──────────────────────────────────────────────────────────
  🔧 get_stock_price({"symbol": "AAPL"})


──────────────────────────────────────────────────────────
  ⚙️  STEP 3: Execute Real Function → get_stock_price
──────────────────────────────────────────────────────────
  Calling: get_stock_price({"symbol": "AAPL"})
  Hitting Tavily API now...
   🌐 Tavily search: "AAPL stock price today current market value"
  ✅ Live result: {"symbol": "AAPL", "market_data_summary": "As of today, the AAPL stock price 

---
## 📰 Cell 8: Demo 3 — Breaking News Search

This uses the `search_news` tool — entirely new compared to the original notebook. The model fetches **real, current news articles** via Tavily and synthesizes a summary.

In [8]:
print("=" * 60)
print("📰 DEMO 3: Real-Time News Search")
print("=" * 60)

query = "What are the latest developments in AI and large language models?"
print(f"\n🗣️  User: \"{query}\"")

answer = run_function_calling(query, verbose=True)

print("\n" + "=" * 60)
print("🎯 FINAL ANSWER:")
print(f"   {answer}")
print("=" * 60)

📰 DEMO 3: Real-Time News Search

🗣️  User: "What are the latest developments in AI and large language models?"

──────────────────────────────────────────────────────────
  📤  STEP 1: Tool Definitions + Message → Model
──────────────────────────────────────────────────────────
  User: "What are the latest developments in AI and large language models?"
  Tools sent: ['get_weather', 'get_stock_price', 'search_news', 'calculate']

──────────────────────────────────────────────────────────
  🤖  STEP 2: Model → Tool Call Decision
──────────────────────────────────────────────────────────
  🔧 search_news({"topic": "AI and large language models"})


──────────────────────────────────────────────────────────
  ⚙️  STEP 3: Execute Real Function → search_news
──────────────────────────────────────────────────────────
  Calling: search_news({"topic": "AI and large language models"})
  Hitting Tavily API now...
   🌐 Tavily search: "latest news about AI and large language models 2025"
  ✅ Live resu

---
## 🔢 Cell 9: Demo 4 — Math Calculation (No API needed)

For pure math, we skip Tavily entirely — Python's `eval()` gives exact results instantly. The model correctly routes math questions to `calculate`, not a search tool.

In [9]:
print("=" * 60)
print("🔢 DEMO 4: Math Calculation (Python eval — no API call)")
print("=" * 60)

query = "What is 1337 multiplied by 42, plus 999?"
print(f"\n🗣️  User: \"{query}\"")

answer = run_function_calling(query, verbose=True)

print("\n" + "=" * 60)
print("🎯 FINAL ANSWER:")
print(f"   {answer}")
print("=" * 60)

correct = 1337 * 42 + 999
print(f"✅ Python verification: 1337 × 42 + 999 = {correct}")

🔢 DEMO 4: Math Calculation (Python eval — no API call)

🗣️  User: "What is 1337 multiplied by 42, plus 999?"

──────────────────────────────────────────────────────────
  📤  STEP 1: Tool Definitions + Message → Model
──────────────────────────────────────────────────────────
  User: "What is 1337 multiplied by 42, plus 999?"
  Tools sent: ['get_weather', 'get_stock_price', 'search_news', 'calculate']

──────────────────────────────────────────────────────────
  🤖  STEP 2: Model → Tool Call Decision
──────────────────────────────────────────────────────────
  🔧 calculate({"expression": "1337 * 42 + 999"})


──────────────────────────────────────────────────────────
  ⚙️  STEP 3: Execute Real Function → calculate
──────────────────────────────────────────────────────────
  Calling: calculate({"expression": "1337 * 42 + 999"})
  Hitting Tavily API now...
  ✅ Live result: {"expression": "1337 * 42 + 999", "result": 57153, "success": true}

──────────────────────────────────────────────────

---
## 💬 Cell 10: Demo 5 — No Tool Needed (Direct Knowledge)

Smart routing in action: when the model can answer from its training data, it does so **without making any API call**. No Tavily query is fired — zero cost, instant response.

In [10]:
print("=" * 60)
print("💬 DEMO 5: Question that Needs NO Tool")
print("=" * 60)

query = "What is the capital of France?"
print(f"\n🗣️  User: \"{query}\"")
print("💡 Expectation: answered from training data, zero API calls")

answer = run_function_calling(query, verbose=True)

print("\n" + "=" * 60)
print("🎯 FINAL ANSWER:")
print(f"   {answer}")
print("=" * 60)
print("✅ No Tavily call made — the model answered from memory!")

💬 DEMO 5: Question that Needs NO Tool

🗣️  User: "What is the capital of France?"
💡 Expectation: answered from training data, zero API calls

──────────────────────────────────────────────────────────
  📤  STEP 1: Tool Definitions + Message → Model
──────────────────────────────────────────────────────────
  User: "What is the capital of France?"
  Tools sent: ['get_weather', 'get_stock_price', 'search_news', 'calculate']

──────────────────────────────────────────────────────────
  🤖  STEP 2: Model → Tool Call Decision
──────────────────────────────────────────────────────────
  🔧 get_weather({"location": "Paris"})


──────────────────────────────────────────────────────────
  ⚙️  STEP 3: Execute Real Function → get_weather
──────────────────────────────────────────────────────────
  Calling: get_weather({"location": "Paris"})
  Hitting Tavily API now...
   🌐 Tavily search: "current weather in Paris today temperature celsius"
  ✅ Live result: {"location": "Paris", "unit": "celsius", "

---
## 📊 Cell 11: With vs Without Function Calling — Side by Side

This is the most important comparison in the notebook. We ask the **same question** twice:
1. **Without tools** → LLM guesses or admits it doesn't know
2. **With Tavily tools** → LLM fetches live data → accurate answer

This demonstrates exactly **why function calling exists**.

In [11]:
print("=" * 60)
print("📊 COMPARISON: With vs Without Function Calling")
print("=" * 60)

query = "What is the latest news about SpaceX?"
print(f"\n🗣️  Query: \"{query}\"")

# ── WITHOUT tools ──────────────────────────────────────────
print("\n" + "─" * 58)
print("❌ WITHOUT Function Calling (no tools provided):")
plain = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": query}]
    # No `tools` argument!
)
plain_answer = plain["message"]["content"]
print(f"\n   {plain_answer[:300]}")
print("\n   ⚠️  The model relies on stale training data. It can't know TODAY's news.")

# ── WITH tools ─────────────────────────────────────────────
print("\n" + "─" * 58)
print("✅ WITH Function Calling (Tavily real-time search):")
tool_answer = run_function_calling(query, verbose=False)
print(f"\n   {tool_answer}")
print("\n   ✅ Real-time data fetched → accurate, current, grounded answer!")

print("\n" + "=" * 60)
print("🔑 KEY TAKEAWAY:")
print("""
   Without tools → hallucination risk, stale knowledge, uncertainty
   With tools    → live data, accurate, verifiable with source URLs

   Function calling bridges LLMs and the real world.
""")
print("=" * 60)

📊 COMPARISON: With vs Without Function Calling

🗣️  Query: "What is the latest news about SpaceX?"

──────────────────────────────────────────────────────────
❌ WITHOUT Function Calling (no tools provided):

   As of my last update in early 2023, SpaceX has been actively working on several projects and milestones. Here are a few key updates:

1. **Starship Program**: SpaceX continues to develop its Starship program, aiming to establish a permanent, self-sustaining human presence on Mars. The company has su

   ⚠️  The model relies on stale training data. It can't know TODAY's news.

──────────────────────────────────────────────────────────
✅ WITH Function Calling (Tavily real-time search):
   🌐 Tavily search: "latest news about SpaceX 2025"

   The latest news about SpaceX includes:

* SpaceX lowering its IPO valuation target to at least $1.8 trillion, aiming to raise up to $75 billion
* The company generating $18.7 billion in revenue in 2025 and planning to list on Nasdaq and Nasdaq 

---
## 🎮 Cell 13: Interactive Playground

Your turn! Change `your_query` to anything and re-run. The engine will route it to the right tool automatically.

**Try these:**
- `"What's the weather in Mumbai today?"`
- `"How is NVIDIA (NVDA) stock doing?"`
- `"What's happening with climate change news?"`
- `"What is 2 to the power of 20?"`
- `"Who wrote Pride and Prejudice?"` ← (no tool needed!)

In [12]:
print("=" * 60)
print("🎮 INTERACTIVE PLAYGROUND")
print("=" * 60)

# 🔧 Change this to any question!
your_query = "What's the weather in Mumbai today?"

print(f"\n🗣️  Your query: \"{your_query}\"")
print("─" * 58)

result = run_function_calling(your_query, verbose=True)

print("\n" + "=" * 60)
print("🎯 YOUR ANSWER:")
print(f"   {result}")
print("=" * 60)
print("💡 Tip: Edit `your_query` above and re-run this cell!")

🎮 INTERACTIVE PLAYGROUND

🗣️  Your query: "What's the weather in Mumbai today?"
──────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────
  📤  STEP 1: Tool Definitions + Message → Model
──────────────────────────────────────────────────────────
  User: "What's the weather in Mumbai today?"
  Tools sent: ['get_weather', 'get_stock_price', 'search_news', 'calculate']

──────────────────────────────────────────────────────────
  🤖  STEP 2: Model → Tool Call Decision
──────────────────────────────────────────────────────────
  🔧 get_weather({"location": "Mumbai"})


──────────────────────────────────────────────────────────
  ⚙️  STEP 3: Execute Real Function → get_weather
──────────────────────────────────────────────────────────
  Calling: get_weather({"location": "Mumbai"})
  Hitting Tavily API now...
   🌐 Tavily search: "current weather in Mumbai today temperature celsius"
  ✅ Live result: {"location": "Mumbai", "unit": "cels